# Workshop - Clasificarea supraviețuirii pe Titanic cu Naive Bayes

În acest notebook vei aplica independent pașii învățați în workshop-ul anterior.

Dataset-ul Titanic va fi încărcat, iar datele vor fi împărțite în train și test.

Pornind de la `X_train`, `X_test`, `y_train` și `y_test`, vei construi singur întregul pipeline de clasificare:

- analizarea coloanelor;
- alegerea feature-urilor;
- tratarea valorilor lipsă;
- transformarea coloanelor categorice;
- normalizarea datelor;
- alegerea unui model Naive Bayes potrivit;
- antrenarea modelului;
- obținerea predicțiilor;
- evaluarea rezultatelor;
- afișarea matricei de confuzie;
- interpretarea erorilor modelului.


## 1. Importurile necesare

Importurile de bază sunt deja pregătite.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


## 2. Încărcarea dataset-ului Titanic

Vom folosi dataset-ul Titanic.

Fiecare rând reprezintă un pasager, iar coloana `survived` indică dacă acesta a supraviețuit:

- `0` - nu a supraviețuit;
- `1` - a supraviețuit.


In [3]:
dataset_url = (
    "https://raw.githubusercontent.com/"
    "mwaskom/seaborn-data/master/titanic.csv"
)

dataset = pd.read_csv(
    dataset_url
)

dataset.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 3. Inspecție rapidă

Verificăm dimensiunea dataset-ului, tipurile coloanelor, valorile lipsă și distribuția target-ului.


In [4]:
print(
    "Forma dataset-ului:",
    dataset.shape,
)

dataset.info()


Forma dataset-ului: (891, 15)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    str    
 8   class        891 non-null    str    
 9   who          891 non-null    str    
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    str    
 12  embark_town  889 non-null    str    
 13  alive        891 non-null    str    
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), str(7)
memory usage: 92.4 KB


In [5]:
print(
    "Valori lipsă pe coloană:"
)

display(
    dataset.isna().sum()
)

print(
    "\nDistribuția target-ului:"
)

display(
    dataset["survived"].value_counts()
)


Valori lipsă pe coloană:


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


Distribuția target-ului:


survived
0    549
1    342
Name: count, dtype: int64

## 4. Separarea feature-urilor de target

Coloana `survived` este target-ul pe care vrem să îl prezicem.

Toate celelalte coloane sunt păstrate momentan în `X`. Alegerea feature-urilor și preprocessing-ul vor face parte din task.


In [7]:
X = dataset.drop(
    columns=["survived"]
)

y = dataset["survived"]


## 5. Împărțirea în train și test

Folosim:

- `80%` din date pentru train;
- `20%` din date pentru test;
- `random_state=42`;
- `stratify=y`, pentru a păstra aproximativ aceeași proporție a claselor în ambele subseturi.

Preprocessing-ul trebuie învățat folosind doar datele de train și apoi aplicat datelor de test.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(
    "Forma X_train:",
    X_train.shape,
)

print(
    "Forma X_test:",
    X_test.shape,
)

print(
    "Forma y_train:",
    y_train.shape,
)

print(
    "Forma y_test:",
    y_test.shape,
)


Forma X_train: (712, 14)
Forma X_test: (179, 14)
Forma y_train: (712,)
Forma y_test: (179,)


# Task final - Construirea modelului pentru Titanic

Pornind de la `X_train`, `X_test`, `y_train` și `y_test`, construiește singur întregul pipeline de clasificare.

## Cerințe

### 1. Analizarea și pregătirea datelor

1. Inspectează coloanele disponibile.
2. Alege feature-urile pe care le consideri relevante.
3. Elimină coloanele care nu pot fi folosite direct sau care oferă informație redundantă.
4. Tratează valorile lipsă folosind statistici calculate numai din train set.
5. Aplică aceleași valori de înlocuire și datelor de test.
6. Transformă coloanele categorice în valori numerice.
7. Asigură-te că train și test au exact aceleași coloane după transformare.
8. Verifică dacă mai există valori lipsă.

### 2. Normalizarea datelor

9. Normalizează feature-urile în intervalul `0–1`.
10. Folosește `MinMaxScaler`.
11. Apelează `fit_transform()` numai pentru datele de train.
12. Apelează `transform()` pentru datele de test.
13. Verifică valorile minime și maxime după normalizare.

### 3. Construirea modelului

14. Alege o variantă Naive Bayes potrivită datelor.
15. Creează modelul.
16. Antrenează modelul folosind datele normalizate de train.
17. Obține predicțiile pentru datele normalizate de test.

### 4. Evaluarea modelului

18. Calculează accuracy.
19. Afișează accuracy în format zecimal și în procente.
20. Afișează raportul de clasificare.
21. Calculează și afișează matricea de confuzie.
22. Folosește label-urile:
    - `0 - Nu a supraviețuit`;
    - `1 - A supraviețuit`.

### 5. Interpretarea rezultatelor

23. Afișează câteva exemple cu valoarea reală și predicția modelului.
24. Găsește predicțiile greșite.
25. Afișează numărul total de predicții greșite.
26. Explică pe scurt ce observi în matricea de confuzie.

## Întrebări

1. Ce feature-uri ai ales și de ce?
2. Cum ai tratat valorile lipsă?
3. Ce variantă Naive Bayes ai folosit?
4. Ce accuracy ai obținut?
5. Modelul identifică mai bine pasagerii care au supraviețuit sau pe cei care nu au supraviețuit?
6. Ce ai putea modifica pentru a îmbunătăți rezultatele?


In [ ]:
# 1. Alegem feature-urile

selected_features = ...

X_train_selected = ...
X_test_selected = ...


# 2. Tratăm valorile lipsă

...


# 3. Transformăm coloanele categorice

X_train_encoded = ...
X_test_encoded = ...


# 4. Ne asigurăm că train și test au aceleași coloane

...


# 5. Verificăm datele pregătite

print(
    "Forma train:",
    ...
)

print(
    "Forma test:",
    ...
)

print(
    "Valori lipsă în train:",
    ...
)

print(
    "Valori lipsă în test:",
    ...
)


# 6. Normalizăm datele în intervalul 0-1

scaler = ...

X_train_normalized = ...
X_test_normalized = ...

print(
    "Valoarea minimă în train:",
    ...
)

print(
    "Valoarea maximă în train:",
    ...
)


# 7. Construim și antrenăm modelul

model = ...

model.fit(
    ...,
    ...,
)

predictions = ...


# 8. Calculăm și afișăm accuracy

accuracy = ...

print(
    "Accuracy:",
    ...
)

print(
    "Accuracy în procente:",
    ...
)


# 9. Afișăm raportul de clasificare

print(
    classification_report(
        ...,
        ...,
    )
)


# 10. Construim și afișăm matricea de confuzie

confusion = ...

display_confusion = ConfusionMatrixDisplay(
    confusion_matrix=...,
    display_labels=[
        "Nu a supraviețuit",
        "A supraviețuit",
    ],
)

display_confusion.plot(
    cmap="Blues",
    values_format="d",
)

plt.title(
    "Matricea de confuzie — Titanic"
)

plt.show()


# 11. Afișăm câteva predicții

results = pd.DataFrame({
    "Real": ...,
    "Predicție": ...,
})

display(
    results.head(10)
)


# 12. Găsim predicțiile greșite

wrong_predictions = ...

print(
    "Număr de predicții greșite:",
    ...
)

display(
    ...
)


<details>
<summary><strong>Hint</strong></summary>

Poți începe cu un set simplu de feature-uri:

<pre><code>selected_features = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked",
]</code></pre>

Calculează valorile de înlocuire folosind doar train set-ul:

<pre><code>age_median = X_train_selected["age"].median()
fare_median = X_train_selected["fare"].median()
embarked_mode = X_train_selected["embarked"].mode()[0]</code></pre>

Aplică aceleași valori pentru train și test.

Transformă variabilele categorice folosind:

<pre><code>pd.get_dummies(
    dataframe,
    columns=["sex", "embarked"],
    dtype=int,
)</code></pre>

Pentru a obține aceleași coloane în train și test:

<pre><code>X_train_encoded, X_test_encoded = (
    X_train_encoded.align(
        X_test_encoded,
        join="left",
        axis=1,
        fill_value=0,
    )
)</code></pre>

Normalizează datele astfel:

<pre><code>scaler = MinMaxScaler()

X_train_normalized = scaler.fit_transform(
    X_train_encoded
)

X_test_normalized = scaler.transform(
    X_test_encoded
)</code></pre>

Pentru acest tip de date poți începe cu:

<pre><code>model = GaussianNB()</code></pre>

Predicțiile greșite pot fi găsite cu:

<pre><code>wrong_mask = predictions != y_test.to_numpy()</code></pre>

</details>


<details>
<summary><strong>Soluție</strong></summary>

<pre><code># 1. Alegem feature-urile

selected_features = [
    "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked",
]

X_train_selected = X_train[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

# 2. Tratăm valorile lipsă

age_median = X_train_selected["age"].median()
fare_median = X_train_selected["fare"].median()
embarked_mode = X_train_selected["embarked"].mode()[0]

for frame in (X_train_selected, X_test_selected):
    frame["age"] = frame["age"].fillna(age_median)
    frame["fare"] = frame["fare"].fillna(fare_median)
    frame["embarked"] = frame["embarked"].fillna(embarked_mode)

# 3. Transformăm coloanele categorice

X_train_encoded = pd.get_dummies(
    X_train_selected,
    columns=["sex", "embarked"],
    dtype=int,
)

X_test_encoded = pd.get_dummies(
    X_test_selected,
    columns=["sex", "embarked"],
    dtype=int,
)

# 4. Ne asigurăm că train și test au aceleași coloane

X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0,
)

# 5. Verificăm datele pregătite

print("Forma train:", X_train_encoded.shape)
print("Forma test:", X_test_encoded.shape)
print("Valori lipsă în train:", X_train_encoded.isna().sum().sum())
print("Valori lipsă în test:", X_test_encoded.isna().sum().sum())

# 6. Normalizăm datele în intervalul 0-1

scaler = MinMaxScaler()
X_train_normalized = scaler.fit_transform(X_train_encoded)
X_test_normalized = scaler.transform(X_test_encoded)

print("Valoarea minimă în train:", X_train_normalized.min())
print("Valoarea maximă în train:", X_train_normalized.max())

# 7. Construim și antrenăm modelul

model = GaussianNB()
model.fit(X_train_normalized, y_train)
predictions = model.predict(X_test_normalized)

# 8. Calculăm și afișăm accuracy

accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)
print("Accuracy în procente:", accuracy * 100)

# 9. Afișăm raportul de clasificare

print(classification_report(y_test, predictions))

# 10. Construim și afișăm matricea de confuzie

confusion = confusion_matrix(y_test, predictions)

display_confusion = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=["Nu a supraviețuit", "A supraviețuit"],
)

display_confusion.plot(cmap="Blues", values_format="d")
plt.title("Matricea de confuzie — Titanic")
plt.show()

# 11. Afișăm câteva predicții

results = pd.DataFrame({
    "Real": y_test.to_numpy(),
    "Predicție": predictions,
})

display(results.head(10))

# 12. Găsim predicțiile greșite

wrong_mask = predictions != y_test.to_numpy()
wrong_predictions = results[wrong_mask]

print("Număr de predicții greșite:", len(wrong_predictions))
display(wrong_predictions.head(10))</code></pre>

</details>
